INSTALLAZIONE LIBRERIE

In [1]:
pip install tsfel

Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Users\admin\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


In [2]:
pip install imbalanced-learn

You should consider upgrading via the 'c:\Users\admin\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


In [3]:
pip install seaborn


You should consider upgrading via the 'c:\Users\admin\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


In [4]:
pip install matplotlib


Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Users\admin\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


IMPORT LIBRERIE

In [ ]:
import os
import zipfile
import pandas as pd
import numpy as np
import random
import tsfel
from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.metrics import accuracy_score, f1_score
from imblearn.pipeline import Pipeline
from imblearn.combine import SMOTEENN
from sklearn.metrics import make_scorer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

Load Load input_matrix_train e input_matrix_test

In [6]:
input_matrix_train = pd.read_csv('input_matrix_train.csv')

print("Matrice di train caricata:")
print(input_matrix_train)

input_matrix_test = pd.read_csv('input_matrix_test.csv')

print("Matrice di test caricata:")
print(input_matrix_test)




Matrice di train caricata:
       Ax_Absolute energy  Ax_Area under the curve  Ax_Autocorrelation  \
0               26.045873                 0.436424                12.0   
1               25.118106                 0.402989                15.0   
2                9.460307                 0.245629                 3.0   
3               18.140502                 0.324215                 4.0   
4               45.607684                 0.568559                 6.0   
...                   ...                      ...                 ...   
22558           28.731141                 0.457433                 2.0   
22559           27.840389                 0.438950                 2.0   
22560           25.475956                 0.451092                 3.0   
22561           25.331801                 0.459016                 3.0   
22562           18.020646                 0.358108                 4.0   

       Ax_Average power  Ax_Centroid  Ax_ECDF Percentile Count_0  \
0             26

In [8]:
pip install xgboost


Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Users\admin\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


Preprocessing dati (scaling, clipping e smooting)

In [26]:
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from xgboost import XGBClassifier



y = input_matrix_train["target"]
print("Valori unici di y (attività):", y.unique())
print("Occorrenze per ciascun attività:", y.value_counts())
print("Numero di righe e colonne di y:", y.shape)
groups = input_matrix_train["user"]
print("Valori unici di groups (user):", groups.unique())
print("Numero di righe e colonne di groups:", groups.shape)
X = input_matrix_train.drop(columns=['target', 'user','timestamp'])
print("Numero di righe e colonne di X:", X.shape)
print("Prime 10 righe della matrice X")
print(X.head(10))


# Preparazione dei dati di test
y_test = input_matrix_test["target"]
X_test = input_matrix_test.drop(columns=["target", "timestamp"])  # Rimuovi colonne non necessarie
print("Classi uniche in y_test:", sorted(y_test.unique()))
print("Numero di classi in y_test:", len(y_test.unique()))





Valori unici di y (attività): [ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24
 25 26 27 28 29]
Occorrenze per ciascun attività: target
1     2188
11    1767
12    1551
2     1447
4     1257
3     1213
10    1202
8     1147
14    1138
13    1048
5      990
9      738
7      672
27     664
23     561
18     538
6      520
29     451
25     439
21     423
19     380
28     377
24     364
22     335
26     335
15     314
20     209
17     151
16     144
Name: count, dtype: int64
Numero di righe e colonne di y: (22563,)
Valori unici di groups (user): [ 1 10 11 12 13 14 16 17 19  2 20 22 23 24 25  4  5  6  8  9]
Numero di righe e colonne di groups: (22563,)
Numero di righe e colonne di X: (22563, 1560)
Prime 10 righe della matrice X
   Ax_Absolute energy  Ax_Area under the curve  Ax_Autocorrelation  \
0           26.045873                 0.436424                12.0   
1           25.118106                 0.402989                15.0   
2            9.460307       

XGBoost Model

In [ ]:
from imblearn.pipeline import Pipeline  # Usare il Pipeline di imblearn
from imblearn.combine import SMOTEENN  # SMOTEENN di imblearn
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder

# Creare un LabelEncoder per trasformare le etichette da 1-29 a 0-28
label_encoder = LabelEncoder()

# Trasformare le etichette y e y_test
y_encoded = label_encoder.fit_transform(y)
y_test_encoded = label_encoder.transform(y_test)

# Creare e allenare il modello XGBoost come prima
clf = XGBClassifier(
    max_depth=3,
    learning_rate=0.1,
    n_estimators=500,
    objective='multi:softmax',  # Multi-classe
    num_class=29,               # 29 classi (da 0 a 28)
    booster='gbtree'
)

# Creare la pipeline con SMOTEENN
pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),   # Imputazione dei valori mancanti
    ('scaler', StandardScaler()),                  # Standardizzazione
    ('smoteenn', SMOTEENN()),                      # SMOTEENN per bilanciamento
    ('classifier', clf)                            # Classificatore XGBoost
])

# Addestramento della pipeline
pipeline.fit(X, y_encoded)  # Usa y_encoded

# Predizioni
y_pred_encoded = pipeline.predict(X_test)

# Invertire la trasformazione delle etichette per ottenere i valori originali
y_pred = label_encoder.inverse_transform(y_pred_encoded)
y_test_original = label_encoder.inverse_transform(y_test_encoded)

# Valutazione del modello
print("Classification Report:\n")
print(classification_report(y_test_original, y_pred, zero_division=0))

# Confusion Matrix
conf_matrix = confusion_matrix(y_test_original, y_pred)
plt.figure(figsize=(12, 8))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues')
plt.title("Confusion Matrix")
plt.xlabel("Predicted Class")
plt.ylabel("True Class")
plt.show()

# Importanza delle feature
if hasattr(clf, "feature_importances_"):
    feature_importances = pd.Series(clf.feature_importances_, index=X.columns)
    feature_importances.nlargest(10).plot(kind='barh')
    plt.title("Feature Importance")
    plt.show()


SALVATAGGIO MIGLIOR MODELLO E MIGLIORI RISULTATI 

In [ ]:
import pickle
import json


In [ ]:
# Salvare miglior modello con pickle
#apri file best_rf_model.pkl in modalità scrittura binaria
with open("best_rf_model.pkl", "wb") as f:
    pickle.dump(best_rf_model, f) #primo parametro=oggetto da serializare, f=file aperto in modalità scrittura binaria dove verrà salvata la versione serializzata di best_rf_model

# Salva i migliori parametri e il punteggio in un file JSON
best_results = {
    "best_params": rf_cv.best_params_,
    "best_score": best_score
}
with open("best_rf_results.json", "w") as f: #apre un file in modalità scrittura
    json.dump(best_results, f) #Serializza (dump) il dizionario best_results in formato JSON e lo scrive nel file.
print("Modello e risultati salvati correttamente.")


In [ ]:
# Carica il modello
with open("best_rf_model.pkl", "rb") as f:
    loaded_model = pickle.load(f)

# Carica i risultati
with open("best_rf_results.json", "r") as f:
    loaded_results = json.load(f)

print("Modello e risultati caricati correttamente.")
print(f"Migliori parametri: {loaded_results['best_params']}")
print(f"Best F1 score: {loaded_results['best_score']:.2f}")


TRAINING FINALE SU TUTTO IL DATASET DI TRAIN CON IL MIGLIOR MODELLO E I MIGLIORI PARAMETRI 

In [ ]:
from sklearn.ensemble import RandomForestClassifier
import pickle
import json
RANDOM_STATE=18
# Carica i migliori parametri dal file JSON
with open("best_rf_results.json", "r") as f:
    loaded_results = json.load(f)

best_params = loaded_results["best_params"]  # Recupera i migliori parametri
print("Migliori parametri caricati:", best_params)

# Preparazione dei dati
y = input_matrix_train["target"]
X = input_matrix_train.drop(columns=["target", "user","timestamp"])  # Rimuovi le colonne non necessarie

# Crea la pipeline con SMOTE-ENN e RandomForestClassifier
steps = [
    ('smoteenn', SMOTEENN(random_state=RANDOM_STATE)),
    ('classifier', RandomForestClassifier(
        n_estimators=best_params["classifier__n_estimators"],
        max_features=best_params["classifier__max_features"],
        max_depth=best_params["classifier__max_depth"],
        criterion=best_params["classifier__criterion"],
        random_state=RANDOM_STATE
    ))
]
pipeline = Pipeline(steps=steps)

# Addestra il modello sui dati di training
print("Avvio del training sul dataset completo con SMOTE-ENN...")
pipeline.fit(X, y)
print("Pipeline addestrata con successo!")

# Salvataggio del modello finale
with open("final_rf_pipeline.pkl", "wb") as f:
    pickle.dump(pipeline, f)

print("Pipeline finale salvata in 'final_rf_pipeline.pkl'.")

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, f1_score
import seaborn as sns
import matplotlib.pyplot as plt

# Previsioni sui dati di addestramento
y_train_pred = pipeline.predict(X)

# Calcolare l'F1-score sui dati di addestramento
f1 = f1_score(y, y_train_pred, average='macro')
print(f"F1-score sui dati di addestramento: {f1:.2f}")

# Stampa il classification report
print("Classification Report sui dati di addestramento:")
print(classification_report(y, y_train_pred))

# Calcolare la matrice di confusione
conf_matrix = confusion_matrix(y, y_train_pred)

# Visualizzare la matrice di confusione
plt.figure(figsize=(15, 10))
sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues", xticklabels=y.unique(), yticklabels=y.unique())
plt.title("Matrice di Confusione - Training Data")
plt.xlabel("Predizioni")
plt.ylabel("Valori Real")
plt.show()


PREPARAZIONE DATI DI TEST PER VALUTAZIONE FINALE

In [ ]:
#ho aggiunto lable 
for trace in datasetTracesTestFeatExtr:
    trace['TraceDataFeatExtr']['timestamp'] = [1 + i * 0.5 for i in range(len(trace['TraceDataFeatExtr']))]
    trace['TraceDataFeatExtr']['target']=trace['TraceIDFeatExtr'].split('_')[3] #creo una nuova colonna target a cui associo il valore di activity che sta nel traceid

    # Aggiorno la colonna 'target' per gestire l'attività '7' -> '07'
    trace['TraceDataFeatExtr']['target'] = trace['TraceDataFeatExtr']['target'].replace('7', '07')
    
print(datasetTracesTestFeatExtr[0]["TraceDataFeatExtr"])

#concatenato tutti i vari Dataframe =>ottengo un'unica grande matrice
input_matrix_test = pd.concat([item['TraceDataFeatExtr'] for item in datasetTracesTestFeatExtr], ignore_index=True)
print("dimensioni input_matrix:", input_matrix_test.shape)
print(input_matrix_test.head(10))

SALVATAGGIO INPUT MATRIX TEST

In [ ]:
# Salva la matrice in un file CSV
input_matrix_test.to_csv('input_matrix_test.csv', index=False)

DATI DI TEST, CONFUSION MATRIX E REPORT DI CLASSIFICAZIONE

In [ ]:
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import pickle

# Caricamento pipeline salvata
with open("final_rf_pipeline.pkl", "rb") as f:
    pipeline = pickle.load(f)
print("Pipeline caricata con successo.")

# Preparazione dei dati di test
y_test = input_matrix_test["target"]
X_test = input_matrix_test.drop(columns=["target", "timestamp"])  # Rimuovi colonne non necessarie
print("Classi uniche in y_test:", sorted(y_test.unique()))
print("Numero di classi in y_test:", len(y_test.unique()))



# uso del miglior modello per fare previsioni sui dati 
y_pred = pipeline.predict(X_test) #y_pred= array che contiene le etichette previste dal modello per ciascun esempio in X


# Calcolare l'F1-score sui dati di addestramento
f1 = f1_score(y_test, y_pred, average='macro')
print(f"F1-score sui dati di test: {f1:.2f}")

# Stampa il classification report
print("Classification Report sui dati di test:")
print(classification_report(y_test, y_pred))

# Calcolare la matrice di confusione
cm = confusion_matrix(y_test, y_pred)

# Visualizzare la matrice di confusione
plt.figure(figsize=(15, 10))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=y_test.unique(), yticklabels=y_test.unique())
plt.title("Matrice di Confusione - Test Data")
plt.xlabel("Predizioni")
plt.ylabel("Valori Real")
plt.show()








In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report

#report di classificazione
report_dict = classification_report(y_test, y_pred, output_dict=True)

#  precision, recall, F1-score per ciascuna classe
class_names = list(report_dict.keys())[:-3]  
precision = [report_dict[c]['precision'] for c in class_names]
recall = [report_dict[c]['recall'] for c in class_names]
f1_score = [report_dict[c]['f1-score'] for c in class_names]

# Creazione dei grafici a barre per Precision, Recall, e F1-score
x = np.arange(len(class_names))
width = 0.25

plt.figure(figsize=(12, 8))
plt.bar(x - width, precision, width, label='Precision', color='skyblue')
plt.bar(x, recall, width, label='Recall', color='lightgreen')
plt.bar(x + width, f1_score, width, label='F1-Score', color='salmon')

plt.xticks(x, class_names, rotation=45, fontsize=12)
plt.title('Precision, Recall, and F1-Score per Class', fontsize=16)
plt.xlabel('Class', fontsize=14)
plt.ylabel('Score', fontsize=14)
plt.legend(fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()
